# 02 — Embedding Models

Embeddings are the backbone of semantic search in a RAG pipeline.  
The **embedding model** you choose determines how well your system understands the meaning of text, and directly affects retrieval quality.

In this notebook you will:

1. Understand what **text embeddings** are and why they matter
2. Load and compare **different embedding models** from Sentence-Transformers
3. Measure **embedding dimensions**, **speed**, and **similarity behaviour**
4. Store embeddings in Qdrant and observe how model choice affects retrieval

---

## 1 · Setup

> **Prerequisite:** Make sure Qdrant is running at `localhost:6333`.  
> You can start it with `docker compose up qdrant -d` from the project root.

In [ ]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import hashlib
import uuid

In [ ]:
# Connect to local Qdrant instance
qdrant = QdrantClient(host="localhost", port=6333)
print("Qdrant collections:", [c.name for c in qdrant.get_collections().collections])

---
## 2 · Sample Data

We define sample sentences that cover different topics so we can observe how
models distinguish between them.

In [ ]:
# Sentences grouped by topic — useful for evaluating how well a model
# captures semantic similarity within and across topics.

SENTENCES = [
    # Topic A: AI policy & funding
    "Germany's Federal AI Strategy invests 500 million euros per year in AI research.",
    "The government's AI strategy aims to make Europe a leading centre for artificial intelligence.",
    # Topic B: RAG systems
    "Retrieval-Augmented Generation combines a retriever with a language model to produce grounded answers.",
    "In a RAG pipeline, relevant documents are retrieved and included in the LLM prompt.",
    # Topic C: AI service centre
    "The AI Service Centre offers workshops on deep learning and RAG systems.",
    "Researchers can attend AI advisory hours to get help with machine learning projects.",
    # Topic D: Unrelated / control
    "The weather in Berlin is cold and rainy in November.",
    "A good recipe for apple pie requires fresh Granny Smith apples.",
]

QUERIES = [
    "How much money does Germany spend on AI?",
    "What is retrieval-augmented generation?",
    "Where can I get help with my machine learning project?",
]

print(f"{len(SENTENCES)} sentences, {len(QUERIES)} queries")

---
## 3 · Loading Embedding Models

We compare three models with different sizes and capabilities:

| Model | Dimensions | Parameters | Notes |
|-------|-----------|------------|-------|
| `all-MiniLM-L6-v2` | 384 | 22M | Fast, lightweight — used by the workshop backend |
| `all-mpnet-base-v2` | 768 | 110M | Higher quality, slower |
| `multi-qa-mpnet-base-dot-v1` | 768 | 110M | Optimised for question-answering retrieval |

In [ ]:
MODEL_CONFIGS = {
    "MiniLM": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "dim": 384,
    },
    "MPNet": {
        "name": "sentence-transformers/all-mpnet-base-v2",
        "dim": 768,
    },
    "MultiQA-MPNet": {
        "name": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
        "dim": 768,
    },
}

loaded_models = {}
for label, cfg in MODEL_CONFIGS.items():
    print(f"Loading {label} ({cfg['name']})...")
    loaded_models[label] = SentenceTransformer(cfg["name"], device="cpu")

print("\nAll models loaded.")

---
## 4 · Comparing Embeddings

### 4.1 — Embedding speed

How long does each model take to embed the same set of sentences?

In [ ]:
for label, mdl in loaded_models.items():
    start = time.perf_counter()
    embs = mdl.encode(SENTENCES, convert_to_numpy=True)
    elapsed = time.perf_counter() - start
    print(f"{label:20s}  dim={embs.shape[1]:4d}  time={elapsed:.3f}s  shape={embs.shape}")

### 4.2 — Cosine similarity matrix

For each model, compute pairwise cosine similarity between all sentences.  
Ideally, sentences about the same topic should have high similarity.

In [ ]:
def cosine_similarity_matrix(embeddings: np.ndarray) -> np.ndarray:
    """Compute pairwise cosine similarity."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / norms
    return normed @ normed.T

In [ ]:
for label, mdl in loaded_models.items():
    embs = mdl.encode(SENTENCES, convert_to_numpy=True)
    sim = cosine_similarity_matrix(embs)

    print(f"\n{'='*60}")
    print(f"Cosine similarity matrix — {label}")
    print(f"{'='*60}")

    # Print header
    print(f"{'':>6s}", end="")
    for j in range(len(SENTENCES)):
        print(f"  S{j:d}", end="")
    print()

    for i in range(len(SENTENCES)):
        print(f"S{i:<5d}", end="")
        for j in range(len(SENTENCES)):
            print(f" {sim[i,j]:.2f}", end="")
        print(f"  | {SENTENCES[i][:50]}...")

### 4.3 — Query-to-sentence similarity

For each query, rank the sentences by similarity using each model.  
This shows how model choice affects which chunks would be retrieved.

In [ ]:
for query in QUERIES:
    print(f"\nQuery: \"{query}\"\n")

    for label, mdl in loaded_models.items():
        q_emb = mdl.encode([query], convert_to_numpy=True)
        s_embs = mdl.encode(SENTENCES, convert_to_numpy=True)

        scores = (q_emb @ s_embs.T)[0]
        # Normalize for cosine
        scores = scores / (np.linalg.norm(q_emb) * np.linalg.norm(s_embs, axis=1))

        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

        print(f"  {label}:")
        for rank, (idx, score) in enumerate(ranked[:3]):
            print(f"    #{rank+1}  score={score:.4f}  S{idx}: {SENTENCES[idx][:80]}")
    print()

---
## 5 · Retrieval with Different Models

Let's store the same chunks in **separate Qdrant collections** (one per model) and compare retrieval.

In [ ]:
def make_point_id(label: str, idx: int) -> str:
    raw = f"{label}_{idx}"
    hash_hex = hashlib.sha256(raw.encode()).hexdigest()
    return str(uuid.UUID(hash_hex[:32]))


for label, cfg in MODEL_CONFIGS.items():
    col_name = f"nb02_{label.lower()}"
    mdl = loaded_models[label]

    # (Re-)create collection
    if qdrant.collection_exists(col_name):
        qdrant.delete_collection(col_name)

    qdrant.create_collection(
        collection_name=col_name,
        vectors_config=models.VectorParams(
            size=cfg["dim"],
            distance=models.Distance.COSINE,
        ),
    )

    # Embed and upsert
    embs = mdl.encode(SENTENCES, convert_to_numpy=True)
    points = [
        models.PointStruct(
            id=make_point_id(label, i),
            vector=emb.tolist(),
            payload={"sentence": sent, "index": i},
        )
        for i, (sent, emb) in enumerate(zip(SENTENCES, embs))
    ]
    qdrant.upsert(collection_name=col_name, points=points)
    print(f"Collection '{col_name}': {len(points)} points (dim={cfg['dim']})")

In [ ]:
# Compare retrieval across models for each query
for query in QUERIES:
    print(f"\nQuery: \"{query}\"")
    print("-" * 80)

    for label, cfg in MODEL_CONFIGS.items():
        col_name = f"nb02_{label.lower()}"
        mdl = loaded_models[label]
        q_vec = mdl.encode(query, convert_to_numpy=True).tolist()

        results = qdrant.query_points(
            collection_name=col_name,
            query=q_vec,
            limit=3,
        ).points

        print(f"\n  {label}:")
        for r in results:
            print(f"    score={r.score:.4f}  | {r.payload['sentence'][:80]}")

---
## 6 · Exercises

### Exercise 1 — Add another model

Add a **multilingual** model (e.g. `paraphrase-multilingual-MiniLM-L12-v2`) and test it  
with queries in German. Does it retrieve the correct passages?

In [ ]:
# TODO: Load a multilingual model and test it with German queries

### Exercise 2 — Distance metrics

Qdrant supports `COSINE`, `DOT`, and `EUCLID` distance metrics.  
Create three collections (same model, different metrics) and compare search results.

In [ ]:
# TODO: Compare distance metrics (COSINE vs DOT vs EUCLID)

### Exercise 3 — Embedding visualisation

Use PCA or t-SNE (from `scikit-learn`) to reduce embeddings to 2D and plot them.  
Do sentences about the same topic cluster together? Does the model choice matter?

In [ ]:
# TODO: Visualise embeddings in 2D
# Hint: from sklearn.decomposition import PCA
# Hint: import matplotlib.pyplot as plt

---
## 7 · Cleanup

Delete the collections created by this notebook.

In [ ]:
# Uncomment to delete:
# for label in MODEL_CONFIGS:
#     col_name = f"nb02_{label.lower()}"
#     qdrant.delete_collection(col_name)
#     print(f"Deleted collection '{col_name}'")